[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sensioai/blog/blob/master/028_pytorch_nn/pytorch_nn.ipynb)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Pytorch - Redes Neuronales

In [3]:
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score


In [4]:
# Cargar dataset
data = pd.read_csv('/content/drive/MyDrive/IA/DATASETS/Electric_Vehicle_Population_Data (1).csv')

# Revisar nombres de columnas
print("Columnas:", list(data.columns))
print("Registros:", len(data))

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/IA/DATASETS/Electric_Vehicle_Population_Data (1).csv'

## Modelos secuenciales

La forma más sencilla de definir una `red neuronal` en `Pytorch` es utilizando la clase `Sequentail`. Esta clase nos permite definir una secuencia de capas, que se aplicarán de manera secuencial (las salidas de una capa serán la entrada de la siguiente). Ésto ya lo conocemos de posts anteriores, ya que es la forma ideal de definir un `Perceptrón Multicapa`.

In [ ]:
EVT = "Electric Vehicle Type"
df = data.dropna(subset=[EVT]).copy()

df = df.drop(columns=[c for c in ["VIN (1-10)", "DOL Vehicle ID", "Vehicle Location", "2020 Census Tract", "Postal Code"] if c in df.columns])

import pandas as pd, numpy as np
X = pd.get_dummies(df.drop(columns=[EVT]), dummy_na=True).astype(np.float32)
X = X.replace([np.inf, -np.inf], np.nan).fillna(0.0)

from sklearn.preprocessing import LabelEncoder
lbl = LabelEncoder()
y = lbl.fit_transform(df[EVT].astype(str)).astype(np.int64)

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X.values, y, test_size=0.15, random_state=42, stratify=y
)
X_train = np.nan_to_num(X_train, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
X_test  = np.nan_to_num(X_test,  nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)


In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_test  = scaler.transform(X_test).astype(np.float32)

X_t = torch.from_numpy(X_train).float()
Y_t = torch.from_numpy(y_train).long()

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)
class_weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
w = torch.tensor(class_weights, dtype=torch.float32)
print("Pesos de clase:", w)


Pesos de clase: tensor([0.6281, 2.4518])


In [ ]:
num_classes = len(np.unique(y_train))
D_in, H, D_out = X_train.shape[1], 100, num_classes

model = torch.nn.Sequential(
    torch.nn.Linear(D_in, H),
    torch.nn.ReLU(),
    torch.nn.Linear(H, D_out),
)

In [ ]:
x_prueba = torch.randn(600, D_in)
outputs = model(x_prueba)
print(outputs.shape)

torch.Size([600, 2])


In [ ]:
print(outputs[0][:])

tensor([ 0.3050, -0.0220], grad_fn=<SliceBackward0>)


In [ ]:
model

Sequential(
  (0): Linear(in_features=1421, out_features=100, bias=True)
  (1): ReLU()
  (2): Linear(in_features=100, out_features=2, bias=True)
)

In [ ]:
device = torch.device("cpu")
model = model.to(device)

X_t = torch.from_numpy(X_train).float().to(device)
Y_t = torch.from_numpy(y_train).long().to(device)

# Chequeo de seguridad (evita el error de shapes)
assert model[0].in_features == X_t.shape[1], f"{model[0].in_features} vs {X_t.shape[1]}"


In [ ]:
# función de pérdida y derivada

def softmax(x):
    return torch.exp(x) / torch.exp(x).sum(axis=-1, keepdims=True)

def cross_entropy(logits, targets):
    return (torch.logsumexp(logits, dim=1) - logits[torch.arange(logits.size(0)), targets]).mean()
    loss = cross_entropy(y_pred, Y_t)
    return loss

In [ ]:
print(X)

        Model Year  Electric Range  Base MSRP  Legislative District  \
0           2019.0           220.0        0.0                  15.0   
1           2024.0            21.0        0.0                  35.0   
2           2022.0            26.0        0.0                  32.0   
3           2017.0            14.0        0.0                  30.0   
4           2013.0            75.0        0.0                  40.0   
...            ...             ...        ...                   ...   
257630      2020.0            32.0        0.0                  21.0   
257631      2022.0             0.0        0.0                  48.0   
257632      2019.0            15.0    55700.0                  18.0   
257633      2019.0            25.0        0.0                  40.0   
257634      2025.0             0.0        0.0                  43.0   

        County_Ada  County_Adams  County_Alameda  County_Albemarle  \
0              0.0           0.0             0.0               0.0   
1      

In [ ]:
# bucle entrenamiento
epochs = 50
lr = 0.01
log_each = 10
#l = []
for e in range(1, epochs + 1):
    model.train()
    # forward
    y_pred = model(X_t)
    # loss
    loss = cross_entropy(y_pred, Y_t)
    #l.append(loss.item())

    # ponemos a cero los gradientes
    model.zero_grad()

    # Backprop (calculamos todos los gradientes automáticamente)
    loss.backward()

    # update de los pesos
    with torch.no_grad():
        for param in model.parameters():
            param -= lr * param.grad

    if not e % log_each:
        print(f"Epoch {e}/{epochs} Loss {loss.item():.5f}")

Epoch 10/50 Loss 0.62040
Epoch 20/50 Loss 0.59157
Epoch 30/50 Loss 0.56823
Epoch 40/50 Loss 0.54840
Epoch 50/50 Loss 0.53085


Como puedes observar en el ejemplo, podemos calcular la salida del modelo con una simple línea. Luego calculamos la función de pérdida, y llamando a la función `backward` `Pytorch` se encarga de calcular las derivadas de la misma con respecto a todos los parámetros del modelo automáticamente (si no queremos acumular estos gradientes, nos aseguramos de llamar a la función `zero_grad` para ponerlos a cero antes de calcularlos). Por útlimo, podemos iterar por los parámetros del modelo aplicando la regla de actualización deseada (en este caso usamos `descenso por gradiente`).

In [ ]:
from sklearn.metrics import accuracy_score

def evaluate(x):
    model.eval()
    with torch.no_grad():
        x = x.to(device)
        x = torch.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
        y_pred = model(x)
        y_probas = F.softmax(y_pred, dim=1)
        return torch.argmax(y_probas, dim=1)

y_pred = evaluate(torch.from_numpy(X_test).float().to(device))
accuracy_score(y_test, y_pred.cpu().numpy())

0.8007297003570875

## Optimizadores y Funciones de pérdida

En el ejemplo anterior hemos calculado la función de pérdida y aplicado la regla de optimización de forma manual. Sin embargo, `Pytorch` nos ofrece funcionalidad que nos abstrae estos cálculos ofreciendo además flexibilidad para aplicar diferentes funciones de pérdida o algoritmos de optimización de manera sencilla. Podemos encontrar diferentes funciones de pérdida ya implementadas en el paquete `torch.nn`.

In [ ]:
criterion = torch.nn.CrossEntropyLoss(weight=w)

Mientras que los optimizadores se encuentran en el paquete `torch.optim`

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

In [ ]:
D_in, H, D_out = X_train.shape[1], 100, num_classes

model = torch.nn.Sequential(
    torch.nn.Linear(D_in, H),
    torch.nn.ReLU(),
    torch.nn.Linear(H, D_out),
)

criterion = torch.nn.CrossEntropyLoss(weight=w)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

epochs = 50
log_each = 10
l = []
model.train()
for e in range(1, epochs+1):

    # forward
    y_pred = model(X_t)

    # loss
    loss = criterion(y_pred, Y_t)
    l.append(loss.item())

    # ponemos a cero los gradientes
    optimizer.zero_grad()

    # Backprop (calculamos todos los gradientes automáticamente)
    loss.backward()

    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)

    # update de los pesos
    optimizer.step()

    if not e % log_each:
        print(f"Epoch {e}/{epochs} Loss {np.mean(l):.5f}")


y_pred = evaluate(torch.from_numpy(X_test).float())
print(accuracy_score(y_test, y_pred.cpu().numpy()))

Epoch 10/50 Loss 0.67272
Epoch 10/50 Loss 0.63180
Epoch 20/50 Loss 0.61932
Epoch 20/50 Loss 0.50674
Epoch 30/50 Loss 0.55375
Epoch 30/50 Loss 0.35200
Epoch 40/50 Loss 0.48375
Epoch 40/50 Loss 0.21662
Epoch 50/50 Loss 0.42021
Epoch 50/50 Loss 0.13194
0.9894943849298763


## Modelos custom

Si bien en muchos casos definir una `red neuronal` como una secuencia de capas es suficiente, en otros casos será un factor limitante. Un ejemplo son las redes residuales, en las que no sólo utilizamos la salida de una capa para alimentar la siguiente si no que, además, le sumamos su propia entrada. Este tipo de arquitectura no puede ser definida con la clase `Sequential`, y para ello necesitamos hacer un modelo *customizado*. Para ello, `Pytroch` nos ofrece la siguiente sintaxis.

In [ ]:
# creamos una clase que hereda de `torch.nn.Module`

class ModeloPersonalizado(torch.nn.Module):

    # constructor
    def __init__(self, D_in, H, D_out):

        # llamamos al constructor de la clase madre
        super(ModeloPersonalizado, self).__init__()

        # definimos nuestras capas
        self.fc1 = torch.nn.Linear(D_in, H)
        self.relu = torch.nn.ReLU()
        self.fc2 = torch.nn.Linear(H, D_out)

    # lógica para calcular las salidas de la red
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

En primer lugar, necesitamos definir una nueva clase que herede de la clase `torch.nn.Module`. Esta clase madre aportará toda la funcionalidad esencial que necesita una `red neuronal` (soporte GPU, iterar por sus parámeteros, etc). Luego, en esta clase necesitamos definir mínimos dos funciones:

- `init`: en el constructor llamaremos al constructor de la clase madre y después definiremos todas las capas que querramos usar en la red.
- `forward`: en esta función definimos toda la lógica que aplicaremos desde que recibimos los inputs hasta que devolvemos los outputs.

En el ejemplo anterior simplemente hemos replicado la misma red (puedes conseguir el mismo efecto usando la clase `Sequential`).

In [ ]:
model = ModeloPersonalizado(D_in, 100, num_classes)
# Codigo para saber si el modelo esta votando los datos en las cantidades correctas
x_prueba = torch.randn(500, D_in)
print(x_prueba)
outputs = model(x_prueba)
outputs.shape

tensor([[ 0.7391, -0.2815, -0.3138,  ..., -0.6580, -0.3177,  0.9573],
        [ 1.0633, -0.8609, -0.4446,  ..., -1.5679,  0.6886, -0.2035],
        [ 1.4293, -1.4317, -1.1128,  ...,  1.0105, -0.4688,  0.0094],
        ...,
        [-0.4911,  0.2220,  0.2530,  ...,  0.1642,  1.5468,  0.2568],
        [ 1.2767,  0.0506, -1.6185,  ..., -0.7919, -0.6504,  0.3896],
        [-0.8312, -0.3761,  0.7847,  ..., -0.7336, -0.6387, -2.1563]])


torch.Size([500, 2])

Ahora, podemos entrenar nuestra red de la misma forma que lo hemos hecho anteriormente.

In [ ]:
device = torch.device("cpu")
model = model.to(device)

criterion = torch.nn.CrossEntropyLoss(weight=w)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

epochs = 50       
log_each = 10
l = []
model.train()
for e in range(1, epochs+1):

    # forward
    y_pred = model(X_t)

    # loss
    loss = criterion(y_pred, Y_t)
    l.append(loss.item())

    # ponemos a cero los gradientes
    optimizer.zero_grad()

    # Backprop (calculamos todos los gradientes automáticamente)
    loss.backward()

    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)

    # update de los pesos
    optimizer.step()

    if not e % log_each:
        print(f"Epoch {e}/{epochs} Loss {np.mean(l):.5f}")

y_pred = evaluate(torch.from_numpy(X_test).float())
accuracy_score(y_test, y_pred.cpu().numpy())

Epoch 10/50 Loss 0.67836
Epoch 20/50 Loss 0.62650
Epoch 30/50 Loss 0.56358
Epoch 40/50 Loss 0.49549
Epoch 50/50 Loss 0.43216


0.9906588003933137

Aquí puedes ver otro ejemplo de como definir un `MLP` con conexiones residuales, algo que no podemos hacer simplemente usando un modelo secuencial.

In [ ]:
class ModelCustom2(torch.nn.Module):

    def __init__(self, D_in, H, D_out):
        super(ModelCustom2, self).__init__()
        self.fc1 = torch.nn.Linear(D_in, H)
        self.relu = torch.nn.ReLU()
        self.fc2 = torch.nn.Linear(H, D_out)

    def forward(self, x):
        x1 = self.fc1(x)
        x = self.relu(x1)
        x = self.fc2(x + x1)
        return x

In [ ]:
model = ModelCustom2(D_in, 100, num_classes)

with torch.no_grad():  # reinicio limpio
    torch.nn.init.kaiming_uniform_(model.fc1.weight, a=0.0)
    torch.nn.init.zeros_(model.fc1.bias)
    torch.nn.init.kaiming_uniform_(model.fc2.weight, a=0.0)
    torch.nn.init.zeros_(model.fc2.bias)

criterion = torch.nn.CrossEntropyLoss(weight=w)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

X_t = torch.nan_to_num(X_t, nan=0.0, posinf=0.0, neginf=0.0)
X_t = torch.clamp(X_t, -1e6, 1e6)

epochs = 50
log_each = 10
l = []
model.train()
for e in range(1, epochs+1):

    # forward
    y_pred = model(X_t)

    # loss
    loss = criterion(y_pred, Y_t)
    l.append(loss.item())

    # ponemos a cero los gradientes
    optimizer.zero_grad()

    # Backprop (calculamos todos los gradientes automáticamente)
    loss.backward()

    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)

    # update de los pesos
    optimizer.step()

    if not e % log_each:
        print(f"Epoch {e}/{epochs} Loss {np.mean(l):.5f}")

y_pred = evaluate(torch.from_numpy(X_test).float())
accuracy_score(y_test, y_pred.cpu().numpy())

Epoch 10/200 Loss 1.02080
Epoch 20/200 Loss 0.83857
Epoch 30/200 Loss 0.72122
Epoch 40/200 Loss 0.63724
Epoch 50/200 Loss 0.57336
Epoch 60/200 Loss 0.52274
Epoch 70/200 Loss 0.48141
Epoch 80/200 Loss 0.44688
Epoch 90/200 Loss 0.41752
Epoch 100/200 Loss 0.39219
Epoch 110/200 Loss 0.37007
Epoch 120/200 Loss 0.35054
Epoch 130/200 Loss 0.33315
Epoch 140/200 Loss 0.31756
Epoch 150/200 Loss 0.30349
Epoch 160/200 Loss 0.29071
Epoch 170/200 Loss 0.27905
Epoch 180/200 Loss 0.26835
Epoch 190/200 Loss 0.25849
Epoch 200/200 Loss 0.24938


0.9912280701754386

De esta manera, tenemos mucha flexibilidad para definir nuestras redes.

## Accediendo a las capas de una red

En ocasiones queremos acceder a una capa en particular de nuestra red. Para ello, podemos acceder utilizando su nombre.

In [ ]:
model

ModelCustom2(
  (fc1): Linear(in_features=1421, out_features=100, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=100, out_features=2, bias=True)
)

In [ ]:
model.fc1

Linear(in_features=1421, out_features=100, bias=True)

También podemos acceder directamente a los tensores que contienen los parámetros con las propiedades adecuadas

In [ ]:
model.fc1.weight

Parameter containing:
tensor([[ 0.0055, -0.0196,  0.0142,  ...,  0.0518, -0.0254, -0.0511],
        [-0.0578,  0.0356, -0.0205,  ...,  0.0176, -0.0375,  0.0169],
        [-0.0447,  0.0706,  0.0319,  ...,  0.0037,  0.0301,  0.0299],
        ...,
        [-0.0154, -0.0052, -0.0614,  ..., -0.0038,  0.0459,  0.0046],
        [-0.0468,  0.0157,  0.0089,  ..., -0.0536, -0.0282,  0.0159],
        [ 0.0442,  0.0181, -0.0352,  ...,  0.0598, -0.0378, -0.0300]],
       requires_grad=True)

In [ ]:
model.fc1.bias

Parameter containing:
tensor([ 0.0540, -0.0026,  0.0359,  0.0034, -0.0377, -0.0275, -0.0024, -0.0506,
        -0.0114,  0.0004, -0.0080,  0.0656, -0.0315, -0.0435,  0.0143,  0.0027,
        -0.0237, -0.0294, -0.0058, -0.0127,  0.0221, -0.0306,  0.0373, -0.0010,
        -0.0052,  0.0199, -0.0619, -0.0099,  0.0016, -0.0081,  0.0612,  0.0417,
        -0.0395, -0.0023,  0.0051,  0.0261, -0.0136,  0.0268, -0.0183,  0.0506,
        -0.0404, -0.0021, -0.0460, -0.0178,  0.0068,  0.0423, -0.0291,  0.0227,
         0.0202, -0.0415,  0.0113,  0.0084, -0.0241, -0.0108, -0.0552,  0.0318,
         0.0258,  0.0489, -0.0150,  0.0009, -0.0126, -0.0378,  0.0140, -0.0318,
         0.0324,  0.0136,  0.0227,  0.0009, -0.0088, -0.0232,  0.0288,  0.0129,
         0.0160, -0.0218,  0.0004,  0.0187,  0.0133,  0.0432, -0.0346, -0.0408,
        -0.0033,  0.0459,  0.0417, -0.0065, -0.0172,  0.0059,  0.0066, -0.0485,
         0.0591, -0.0515,  0.0480, -0.0046,  0.0143,  0.0045, -0.0362,  0.0243,
        -0.0298,  

Es posible sobreescribir una capa de la siguiente manera

In [ ]:
model.fc2 = torch.nn.Linear(100, 1)

model

ModelCustom2(
  (fc1): Linear(in_features=1421, out_features=100, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=100, out_features=1, bias=True)
)

Ahora, la capa final de nuestra red tiene solo una salida. Esta nueva capa ha sido inicializada de manera aleatoria, por lo que esta nueva red no nos va a servir de mucho. Sin embargo, podríamos volver a entrenar esta red en otro problema en el que solo necesitemos una salida aprovechando los pesos que ya hemos entrenado anteriormente con el dataset MNIST. Esto es la base del *transfer learning*, una técnica que utilizaremos muchísimo más adelante y la cual explicaremos en detalle.

A continuación encontrarás varios trucos a la hora de crear redes neuronales a partir de otras que te pueden resultar útiles.

In [ ]:
# obtener una lista con las capas de una red

list(model.children())

[Linear(in_features=1421, out_features=100, bias=True),
 ReLU(),
 Linear(in_features=100, out_features=1, bias=True)]

In [ ]:
# crear nueva red a partir de la lista (excluyendo las útlimas dos capa)

new_model = torch.nn.Sequential(*list(model.children())[:-2])
new_model

Sequential(
  (0): Linear(in_features=1421, out_features=100, bias=True)
)

In [ ]:
# crear nueva red a partir de la lista (excluyendo las útlima capa)

new_model = torch.nn.ModuleList(list(model.children())[:-1])
new_model

ModuleList(
  (0): Linear(in_features=1421, out_features=100, bias=True)
  (1): ReLU()
)